In [10]:
import glob
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path.cwd() / "../data"
OUTPUT = Path.cwd() / "../output"
PRICE_DIR  = BASE / "price"
KOSPI_PATH = BASE / "kospi.csv"
OUT_ALL    = OUTPUT / "price_all.csv"
OUT_CLEAN  = OUTPUT / "price_clean.csv"

In [11]:
# -----------------------
# 1) 유틸
# -----------------------
PRICE_KEYS = ["open","high","low","close","volume","vwap"]

def infer_key_from_filename(p: Path) -> str | None:
    name = p.stem.lower()
    for k in PRICE_KEYS:
        if k in name:
            return k
    return None  # (필요하면 기본값 처리)

def read_price_wide_to_long(p: Path, key: str) -> pd.DataFrame:
    """
    파일 구조 예시:
    ,AEGQRD,AIJFBS,AJAAJF,...
    20200102,6256.45,34434.59,...

    첫 열(헤더 공백)은 날짜, 나머지 열은 전부 종목 심볼.
    """
    df = pd.read_csv(p)
    # 첫 컬럼명이 비어있거나 'Unnamed: 0'일 가능성 → date로 이름 부여
    first_col = df.columns[0]
    if not first_col or first_col.startswith("Unnamed"):
        df = df.rename(columns={first_col: "date"})
    else:
        # 혹시 'date'가 이미 들어있다면 그대로 사용
        if first_col.lower() != "date":
            df = df.rename(columns={first_col: "date"})

    # 날짜 파싱
    df["date"] = pd.to_datetime(df["date"].astype(str), errors="coerce")

    # long 변환
    long_df = df.melt(id_vars="date", var_name="symbol", value_name=key)
    # 심볼/값 정리
    long_df["symbol"] = long_df["symbol"].astype(str).str.strip().str.upper()
    long_df[key] = pd.to_numeric(long_df[key], errors="coerce")
    # (date, symbol) 중복 제거(마지막값 우선)
    long_df = long_df.sort_values(["date","symbol"]).drop_duplicates(["date","symbol"], keep="last")
    return long_df

def read_kospi(p: Path) -> pd.DataFrame:
    """
    kospi.csv 구조 예시:
    ,close,high,open,low
    20200102,25920.63,26307.77,26259.38,25894.23
    → 첫 열은 날짜 인덱스, 나머지는 지수 OHLC
    """
    df = pd.read_csv(p)
    first_col = df.columns[0]
    if not first_col or first_col.startswith("Unnamed"):
        df = df.rename(columns={first_col: "date"})
    elif first_col.lower() != "date":
        df = df.rename(columns={first_col: "date"})
    df["date"] = pd.to_datetime(df["date"].astype(str), errors="coerce")

    # 숫자열만 남기고 kospi_ 접두어
    value_cols = [c for c in df.columns if c != "date"]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[["date"] + value_cols].copy()
    df = df.rename(columns={c: f"kospi_{c.lower()}" for c in value_cols})
    return df

def eda_report(df: pd.DataFrame) -> dict:
    rep = {
        "shape": df.shape,
        "date_range": (df["date"].min(), df["date"].max()),
        "n_symbols": df["symbol"].nunique(),
        "dupes(date,symbol)": int(df.duplicated(["date","symbol"]).sum()),
        "na_ratio": df.isna().mean().sort_values(ascending=False).to_dict()
    }
    checks = {}
    if {"low","high"}.issubset(df.columns):
        checks["low>high"] = int(((df["low"] > df["high"]) & df["low"].notna() & df["high"].notna()).sum())
    if {"open","low","high"}.issubset(df.columns):
        checks["open_out_of_band"]  = int((((df["open"]  < df["low"]) | (df["open"]  > df["high"])) &
                                           df["open"].notna() & df["low"].notna() & df["high"].notna()).sum())
    if {"close","low","high"}.issubset(df.columns):
        checks["close_out_of_band"] = int((((df["close"] < df["low"]) | (df["close"] > df["high"])) &
                                           df["close"].notna() & df["low"].notna() & df["high"].notna()).sum())
    if {"vwap","low","high"}.issubset(df.columns):
        checks["vwap_out_of_band"]  = int((((df["vwap"]  < df["low"]) | (df["vwap"]  > df["high"])) &
                                           df["vwap"].notna() & df["low"].notna() & df["high"].notna()).sum())
    if "volume" in df.columns:
        checks["negative_volume"]   = int((df["volume"] < 0).sum())
    rep["logic_checks"] = checks
    return rep

def print_eda(title: str, df: pd.DataFrame, topk: int = 10):
    r = eda_report(df)
    print(f"\n=== {title} ===")
    print("shape:", r["shape"])
    print("date_range:", r["date_range"])
    print("#symbols:", r["n_symbols"])
    print("dupes(date,symbol):", r["dupes(date,symbol)"])
    print("NA ratio (top):", dict(list(r["na_ratio"].items())[:topk]))
    print("logic checks:", r["logic_checks"])

In [12]:


# -----------------------
# 2) price_all 만들기
# -----------------------
paths = sorted(glob.glob(str(PRICE_DIR / "*.csv")))
if not paths:
    raise FileNotFoundError(f"No CSV files in {PRICE_DIR}")

# 각 파일 → long으로 읽고 (date,symbol) 기준으로 가로 병합
from functools import reduce

long_dfs = []
for s in paths:
    p = Path(s)
    key = infer_key_from_filename(p) or "value"
    long_dfs.append(read_price_wide_to_long(p, key))

# (date, symbol) outer merge
price_all = reduce(
    lambda l, r: pd.merge(l, r, on=["date","symbol"], how="outer", suffixes=("","_dup")),
    long_dfs
)

# _dup 정리(앞 열 우선)
dup_cols = [c for c in price_all.columns if c.endswith("_dup")]
for c in dup_cols:
    base = c[:-4]
    if base in price_all.columns:
        price_all[base] = price_all[base].where(~price_all[base].isna(), price_all[c])
price_all = price_all.drop(columns=dup_cols)

# KOSPI 병합(날짜 기준, kospi_* 접두어)
kospi = read_kospi(KOSPI_PATH)
price_all = price_all.merge(kospi, on="date", how="left")

# 저장
price_all = price_all.sort_values(["symbol","date"]).reset_index(drop=True)
price_all.to_csv(OUT_ALL, index=False)
print(f"[Saved] price_all → {OUT_ALL}")

print_eda("EDA BEFORE CLEANING (price_all)", price_all)


[Saved] price_all → /Users/masterj/Documents/GitHub/StockPlay-Data-analysis/src/../output/price_all.csv

=== EDA BEFORE CLEANING (price_all) ===
shape: (429619, 12)
date_range: (Timestamp('2020-01-02 00:00:00'), Timestamp('2024-12-30 00:00:00'))
#symbols: 349
dupes(date,symbol): 0
NA ratio (top): {'vwap': 0.09304756074568396, 'high': 0.08636489540732603, 'low': 0.08636489540732603, 'open': 0.08636489540732603, 'close': 0.08565263640574555, 'volume': 0.08565030876194954, 'date': 0.0, 'symbol': 0.0, 'kospi_close': 0.0, 'kospi_high': 0.0}
logic checks: {'low>high': 0, 'open_out_of_band': 0, 'close_out_of_band': 12303, 'vwap_out_of_band': 0, 'negative_volume': 0}


In [13]:
# price_all은 이미 메모리에 있다고 가정
df = price_all.copy()

# 경로 (필요에 맞게 바꾸세요)
OUT_CLEAN = (Path.cwd() / "../output/price_clean.csv").resolve()

# 1) 정렬
df = df.sort_values(["symbol","date"]).reset_index(drop=True)

# 2) 논리 보정: open/close/vwap를 [low, high] 범위로 클리핑 (둘 다 존재하는 행에 한함)
has_low_high = df[["low","high"]].notna().all(axis=1) if {"low","high"}.issubset(df.columns) else pd.Series(False, index=df.index)
for c in ["open","close","vwap"]:
    if c in df.columns and {"low","high"}.issubset(df.columns):
        df.loc[has_low_high & df[c].notna(), c] = np.clip(df.loc[has_low_high, c],
                                                          df.loc[has_low_high, "low"],
                                                          df.loc[has_low_high, "high"])

# 3) 결측치 처리: 가격열만 심볼별 ffill (volume은 그대로 둠)
price_cols = [c for c in ["open","high","low","close","vwap"] if c in df.columns]
if price_cols:
    df[price_cols] = df.groupby("symbol", group_keys=False)[price_cols].ffill()

# 4) 모든 가격열이 전부 NaN인 행 제거 (심볼/날짜만 있는 빈 행 제거)
if price_cols:
    all_nan = df[price_cols].isna().all(axis=1)
    df = df.loc[~all_nan].copy()

# 5) 중복 제거 (안전)
df = df.drop_duplicates(["date","symbol"], keep="last")

# 6) 파생변수: 종가 기준 수익률
if {"symbol","date","close"}.issubset(df.columns):
    df["ret_1d"]  = df.groupby("symbol", group_keys=False)["close"].pct_change()
    df["ret_5d"]  = df.groupby("symbol", group_keys=False)["close"].pct_change(5)
    df["ret_20d"] = df.groupby("symbol", group_keys=False)["close"].pct_change(20)

# 7) 코스피 수익률 (있을 때만)
if "kospi_close" in df.columns:
    df = df.sort_values("date")
    df["kospi_ret_1d"]  = df["kospi_close"].pct_change()
    df["kospi_ret_5d"]  = df["kospi_close"].pct_change(5)
    df["kospi_ret_20d"] = df["kospi_close"].pct_change(20)

# 8) 간단 EDA(After) 출력
def eda_after(_df):
    out = {
        "shape": _df.shape,
        "date_range": (pd.to_datetime(_df["date"]).min(), pd.to_datetime(_df["date"]).max()),
        "n_symbols": _df["symbol"].nunique(),
        "na_ratio_top": dict(_df.isna().mean().sort_values(ascending=False).head(8))
    }
    return out

print("=== EDA AFTER CLEANING (price_clean) ===")
print(eda_after(df))

# 9) 저장
OUT_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_CLEAN, index=False)
print(f"[Saved] price_clean → {OUT_CLEAN}")


=== EDA AFTER CLEANING (price_clean) ===
{'shape': (392822, 18), 'date_range': (Timestamp('2020-01-02 00:00:00'), Timestamp('2024-12-30 00:00:00')), 'n_symbols': 346, 'na_ratio_top': {'ret_20d': np.float64(0.01761612129666872), 'ret_5d': np.float64(0.00440403032416718), 'ret_1d': np.float64(0.000880806064833436), 'vwap': np.float64(0.0005600500990270402), 'kospi_ret_20d': np.float64(5.091364536609457e-05), 'kospi_ret_5d': np.float64(1.2728411341523642e-05), 'kospi_ret_1d': np.float64(2.5456822683047282e-06), 'kospi_open': np.float64(0.0)}}
[Saved] price_clean → /Users/masterj/Documents/GitHub/StockPlay-Data-analysis/output/price_clean.csv


### 결측치 처리

ffill (전일값으로 보정) → 휴일·비상장일 보완

volume은 ffill 하지 말 것 (0 또는 NaN 유지)

전 종목의 거래일을 통일하려면 outer join 후 NaN → ffill

| 출처                                                         | 내용                                                                                                                         |
| ---------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------- |
| **Hull (2018), *Options, Futures, and Other Derivatives*** | “휴일 및 비거래일은 직전 종가를 사용하여 연속적인 가격 시계열을 구성한다.”                                                                                |
| **Pandas 공식 문서 (Resampling & Time-Series)**                | “For missing observations in financial time series, forward fill is common to preserve continuity without lookahead bias.” |
| **Kaggle Quant Competitions / Bloomberg API Docs**         | 대부분 일자 누락을 `ffill`로 채워서 모델 피처 생성 시 날짜 동기화                                                                                  |
